In [1]:
import sys

assert sys.version_info >= (3, 10)

In [2]:
import torch
from packaging.version import Version

assert Version(torch.__version__) >= Version("2.6.0")

In [3]:
import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

In [4]:
from pathlib import Path

IMAGES_PATH = Path() / "images" / "Damped_Harmonic_Oscillator"
IMAGES_PATH.mkdir(parents=True, exist_ok=True)

def save_fig(fig_id, fig_extension="png", tight_layout=True, resolution=300):
    path = IMAGES_PATH / f"{fig_id}.{fig_extension}"
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)

In [5]:
import deepxde as dde
from deepxde import utils
import numpy as np

dde.config.set_random_seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

Using backend: pytorch
Other supported backends: tensorflow.compat.v1, tensorflow, jax, paddle.
paddle supports more examples now and is recommended.


Setting the backend

In [6]:
dde.config.set_default_float("float64")
print(f"Backend: {dde.backend.backend_name}")

Set the default float type to float64
Backend: pytorch


We will now define the Physical parameters


In [7]:
m = 1.0 # Mass
c = 0.5 #Damping
k  = 4.0 # Stiffness

Now we will setup the Initial Conditions of the Damper

In [8]:
x0 = 1.0 # initial displacement
v0 = 0.0 # initial velocity

Now we will be deriving the parameters required to calculate our exact solution

In [9]:
omega_n = np.sqrt(k / m)
zeta = c / (2 * np.sqrt(m * k))
omega_d = omega_n * np.sqrt(1 - zeta**2)

We will define a function for our exact solution

In [10]:
def exact_solution(t):
    # In the formula we have an exponential term, cosine term and sine term
    exp_term = np.exp(-zeta * omega_n * t)
    cos_term = np.cos(omega_d * t)
    sin_term = (zeta * omega_n / omega_d) * np.sin(omega_d * t)
    return exp_term * (cos_term + sin_term)


We will create a function to calculate the exact velocity which is the derivative of displacement with respect to time

In [11]:
def exact_vx(t):
    exp_term  = np.exp(-zeta * omega_n * t)
    decay_rate = -zeta * omega_n

    x = exact_solution(t)
    dx_dt = decay_rate * x + exp_term * omega_d * (
        -np.sin(omega_d * t) + (zeta * omega_n / omega_d) * np.cos(omega_d *t)
    )
    print(dx_dt)
    return dx_dt

In [12]:
exact_vx(2)

0.8997813813378139


np.float64(0.8997813813378139)

We can also create a function with using autograd in torch but everything has to converted to torch tensors if we are working on "cuda" 

In [ ]:
def exact_vx1(t):
    if isinstance(t, torch.Tensor):
        t_tensor = t.clone().detach().requires_grad_(True)
    else:
        t_tensor = torch.tensor(t, requires_grad=True)
    x = exact_solution(t_tensor)
    x.backward(torch.ones_like(x))
    velocity = t_tensor.grad.cpu().detach()
    print(velocity)
    return velocity


Creating a geometry for our time domain


In [14]:
geom = dde.geometry.TimeDomain(0, 10)

Now let's write a function for the motion which is usually written using newton's second law which is an ODE equation

In [15]:
def ode(t, x):
    dx_dt = dde.grad.jacobian(x, t, i=0, j=0)
    d2w_d2t = dde.grad.hessian(x, t, i=0, j=0)

    residual = m * d2w_d2t + c * dx_dt + k * x
    return residual

specify the initial conditions

In [16]:
ic_displacement = dde.icbc.IC(geom, lambda t: x0, lambda _, on_initial: on_initial)

def velocity_ic(inputs, outputs, X):
    dx_dt = dde.grad.jacobian(outputs, inputs, i=0, j=0)
    return dx_dt - v0

def velocity_ic_location(t, on_initial):
    return on_initial and np.isclose(t[0], 0.0)

ic_velocity = dde.icbc.OperatorBC(
    geom, velocity_ic, velocity_ic_location
)
